Universidad Torcuato Di Tella

Licenciatura en Tecnología Digital\
**Tecnología Digital VI: Inteligencia Artificial**

# **Taller sobre regresión logística con PyTorch**

En esta *notebook* buscaremos implementar una regresión logística con PyTorch, usando las operaciones entre tensores.


### **Consigna 1**

> **Carga de datos**. Cargar los datos del *data set* sobre cáncer de mama.

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

data = load_breast_cancer()
X = data.data
y = data.target

### **Consigna 2**

> **Creación de conjuntos**. Separar de forma aleatoria un 80% de los datos para el entrenamiento y el 20% restante para la evaluación.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size = 0.2,
                                                    random_state = 11)

### **Consigna 3**

> **Preprocesamiento 1**. Convertir los *arrays* a tensores de PyTorch.

In [ ]:
import numpy as np
import torch

X_train_tensor = torch.from_numpy(X_train.astype(np.float32))
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
y_train_tensor = torch.from_numpy(y_train.astype(np.float32))
y_test_tensor = torch.from_numpy(y_test.astype(np.float32))

### **Consigna 4**

> **Preprocesamiento 2**. Estandarizar los *inputs*.

In [ ]:
# Calculamos la media y la desviación estándar.
mean = torch.mean(X_train_tensor, axis = 0)
std = torch.std(X_train_tensor, axis = 0)

# Estandarizamos.
X_train_tensor = (X_train_tensor - mean) / std
X_test_tensor = (X_test_tensor - mean) / std

### **Consigna 5**

> **Definición de funciones**. Definir la función de pérdida y la función sigmoide.

In [ ]:
# Función de pérdida.
def cost_function(y, y_pred):
    epsilon = 1e-15
    return -(((y * torch.log(y_pred + epsilon)) + ((1 - y) * torch.log(1 - y_pred + epsilon))).mean())

In [ ]:
# Función sigmoide.
def sigmoid(x):
    return 1 / (1 + torch.exp(-x))

### **Consigna 6**

> **Entrenamiento**. Completar las siguientes dos funciones para definir la regresión. *Tip*: tener en cuenta *qué operaciones* matemáticas se deben realizar entre los tensores.

In [ ]:
def gradient_descent(X, y, alpha, iterations):
    m, n = X.shape
    weights = torch.zeros(n, dtype = torch.float32)

    for _ in range(iterations):
        z = torch.matmul(X, weights)
        preds = sigmoid(z)
        cost = cost_function(y, preds)

        # Calculamos el gradiente y actualizamos los pesos.
        gradient = torch.matmul(X.t(), (preds - y)) / m
        weights -= alpha * gradient

        if (_ + 1) % 30 == 0:
           print(f'Epoch [{_ + 1} / {_}], loss: {cost:.4f}')

    return weights

In [ ]:
def logistic_regression(X, y, alpha = 0.01, iterations = 1000):
    intercept = torch.ones((X.shape[0], 1), dtype = torch.float32)
    X_new = torch.cat((intercept, X), dim = 1)
    weights = gradient_descent(X_new, y, alpha, iterations)
    return weights

### **Consigna 7**

> **Predicción**. Calcular las predicciones para los datos de evaluación.

In [ ]:
def predict(X, weights):
    intercept = torch.ones((X.shape[0], 1), dtype = torch.float32)
    X_new = torch.cat((intercept, X), dim = 1)
    predictions = sigmoid(torch.matmul(X_new, weights))
    return predictions.round()

In [ ]:
weights = logistic_regression(X_train_tensor, y_train_tensor)
predicted_labels = predict(X_test_tensor, weights)

### **Consigna 8**

> **Evaluación**. Calcular el valor de *accuracy* del modelo.

In [ ]:
print(f'Accuracy: {(predicted_labels.view(-1) == y_test_tensor.float()).sum().item() / y_test_tensor.size(0)}')

Accuracy: 0.9736842105263158
